# Mini RLHF Pipeline — Google Colab

This notebook runs the full **mini RLHF pipeline** (SFT → Reward Model → PPO → Eval) on Google Colab.

## How to run

1. **Enable a GPU** — Go to **Runtime → Change runtime type**, set *Hardware accelerator* to **T4 GPU**, then click **Save**.
2. **Run all cells** — Go to **Runtime → Run all** (or press `Ctrl+F9`).
   All cells will execute automatically in order.
3. *(Optional)* **Enable W&B logging** — Before running, find the *W&B setup* cell below and set `USE_WANDB = True`.

> **That's it!** The notebook clones the repo, installs dependencies, and runs all four training steps.

---

### Pipeline overview

| Step | Script | What it does |
|------|--------|--------------|
| 1 | `1_sft.py` | Supervised Fine-Tuning — fine-tune GPT-2 on `chosen` responses → saves `./sft_model` |
| 2 | `2_reward_model.py` | Reward Model — pairwise Bradley-Terry loss on chosen/rejected pairs → saves `./reward_model` |
| 3 | `3_ppo.py` | PPO Training — optimise policy against the reward model with KL penalty → saves `./ppo_model` |
| 4 | `4_eval.py` | Evaluation — compare GPT-2 base, SFT, and PPO models on reward scores |


## 1. Setup: clone repository & install dependencies

In [ ]:
import os

# NOTE: After the PR is merged into main, change BRANCH to "main".
REPO_URL = "https://github.com/Aksh123100/mini-rlhf.git"
BRANCH = "copilot/run-repo-in-colab"
REPO_DIR = "mini-rlhf"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL}

%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt

## 2. (Optional) Weights & Biases login

Set `USE_WANDB = True` below if you want to log metrics to the W&B web UI.
Leave it as `False` to skip W&B entirely (all training still runs normally).

In [ ]:
USE_WANDB = False  # set to True and run wandb.login() if you want online logging

import os
os.environ["USE_WANDB"] = "1" if USE_WANDB else "0"

if USE_WANDB:
    import wandb
    wandb.login()

## 3. Step 1 — Supervised Fine-Tuning (SFT)

Fine-tunes GPT-2 on the `chosen` responses from the `hh-rlhf` dataset.
The resulting model is saved to `./sft_model`.

In [ ]:
!python 1_sft.py

## 4. Step 2 — Reward Model Training

Loads the SFT checkpoint, adds a scalar head, and trains it with pairwise
Bradley-Terry loss using both `chosen` and `rejected` responses.
The trained reward model is saved to `./reward_model`.

In [ ]:
!python 2_reward_model.py

## 5. Step 3 — PPO Training

Optimises the SFT policy against the learned reward model using PPO
with a KL-divergence penalty against a frozen reference copy of the SFT model.
The final policy is saved to `./ppo_model`.

In [ ]:
!python 3_ppo.py

## 6. Step 4 — Evaluation

Compares GPT-2 base, SFT, and PPO models on 10 prompts from the `hh-rlhf` test split
by scoring their responses with the reward model.

In [ ]:
!python 4_eval.py